#**GE338 Lab 4 — Geographic Modeling — สร้างและตรวจสอบแบบจำลองเชิงพื้นที่**


# 🌊 Flood Mapping in Songkhla Province, Thailand  
## Sentinel-1 SAR + Google Earth Engine

โดยใช้ **จังหวัดสงขลา** เป็นพื้นที่ศึกษา

### สิ่งที่ทำในโน้ตบุ๊กนี้
- กำหนดพื้นที่ศึกษาเป็น **จังหวัดสงขลา**
- ดึงข้อมูล **Sentinel-1 SAR** ช่วงก่อนน้ำท่วมและหลังน้ำท่วม
- คำนวณอัตราส่วน **VV/VH**
- ลดสัญญาณรบกวนแบบ **Refined Lee filter**
- ตรวจจับพื้นที่น้ำท่วมด้วย **change ratio**
- ตัดพื้นที่น้ำถาวรด้วย **JRC Global Surface Water**
- ตัดพื้นที่ลาดชันสูงด้วย **SRTM slope**
- ลบพิกเซลรบกวนขนาดเล็กด้วย **connected pixel filtering**
- คำนวณพื้นที่น้ำท่วมเป็น **ตารางเมตร / ตารางกิโลเมตร / ไร่**
- แสดงผลบนแผนที่และเตรียมส่งออกข้อมูลได้

### แนวคิดหลักของงาน
จังหวัดสงขลาเป็นพื้นที่ชายฝั่งและมีพื้นที่ราบต่ำจำนวนมาก อีกทั้งยังได้รับอิทธิพลจากมรสุมตะวันออกเฉียงเหนือ ทำให้เกิดน้ำท่วมได้บ่อย โดยเฉพาะพื้นที่เมืองและพื้นที่รับน้ำอย่างหาดใหญ่และบริเวณรอบทะเลสาบสงขลา  
SAR เหมาะกับการติดตามน้ำท่วม เพราะสามารถทำงานได้ทั้งกลางวัน กลางคืน และทะลุผ่านเมฆได้

## 📚 ข้อมูลที่ใช้ในงาน

### 1) พื้นที่ศึกษา
ใช้ **Songkhla Province** เป็นขอบเขตการวิเคราะห์

### 2) ข้อมูลภาพดาวเทียม
- **Sentinel-1 GRD**
  - Instrument mode: `IW`
  - Polarization: `VV`, `VH`
  - Orbit: `DESCENDING`
  - Spatial resolution: `10 m`

### 3) ช่วงเวลา
- **ก่อนน้ำท่วม:** 1 ต.ค. 2025 – 31 ต.ค. 2025
- **หลังน้ำท่วม:** 20 พ.ย. 2025 – 30 พ.ย. 2025

### 4) ข้อมูลสนับสนุน
- **JRC Global Surface Water** สำหรับตัดน้ำถาวร
- **SRTM DEM** สำหรับคำนวณความลาดชัน

### 5) หลักเกณฑ์สำคัญ
- ใช้ค่า **change ratio threshold = 1.2**
- ตัดพื้นที่ที่มีความลาดชันมากกว่า **5 องศา**
- ตัดกลุ่มพิกเซลที่เชื่อมต่อกันน้อยกว่า **15 พิกเซล**

In [1]:
# ติดตั้งแพ็กเกจที่จำเป็นใน Colab
!pip -q install earthengine-api geemap

## 🔐 Step 1: Authentication และ Initialize Earth Engine

รันเซลล์นี้ก่อนใช้งานทุกครั้งใน Google Colab  
เมื่อรันแล้ว ระบบจะให้ล็อกอิน Google เพื่อใช้งาน Earth Engine

In [2]:
import ee
import geemap

PROJECT = "ee-suranajkrua165"  # <- เปลี่ยนตรงนี้

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("Earth Engine initialized successfully ✅")

Earth Engine initialized successfully ✅


## 📦 Step 2: Import libraries

In [3]:
import math
import ee
import geemap

## 🗺️ Step 3: กำหนดพื้นที่ศึกษา (ROI)

ในที่นี้ใช้ขอบเขตการปกครองระดับจังหวัดจากชุดข้อมูล **FAO/GAUL/2015/level1** แล้วกรองเฉพาะ **Songkhla**

In [5]:
# Administrative boundary
gaul_level1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
roi = gaul_level1.filter(ee.Filter.eq("ADM1_NAME", "Songkhla"))

# Supporting datasets
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
srtm = ee.Image("USGS/SRTMGL1_003")

# Create interactive map
Map = geemap.Map()
Map.centerObject(roi, 9)

roi_style = {
    "color": "red",
    "fillColor": "00000000",
    "width": 2
}
Map.addLayer(roi.style(**roi_style), {}, "Songkhla Boundary")
Map

Map(center=[6.939060963818664, 100.54501949331903], controls=(WidgetControl(options=['position', 'transparent_…

## 🕒 Step 4: กำหนดช่วงเวลาก่อนน้ำท่วมและหลังน้ำท่วม

In [6]:
before_start = "2025-10-01"
before_end   = "2025-10-31"
after_start  = "2025-11-20"
after_end    = "2025-11-30"

print("Before flood period:", before_start, "to", before_end)
print("After flood period :", after_start, "to", after_end)

Before flood period: 2025-10-01 to 2025-10-31
After flood period : 2025-11-20 to 2025-11-30


## 📡 Step 5: ดึงข้อมูล Sentinel-1 SAR

เงื่อนไขการคัดกรองภาพ:
- อยู่ในพื้นที่จังหวัดสงขลา
- โหมด `IW`
- มี polarization ทั้ง `VV` และ `VH`
- วงโคจร `DESCENDING`
- ความละเอียด `10 เมตร`

จากนั้นสร้างภาพแบบ **median composite** สำหรับช่วงก่อนและหลังน้ำท่วม เพื่อช่วยลดสัญญาณรบกวนและให้ค่ามีความเสถียรมากขึ้น

In [7]:
s1 = ee.ImageCollection("COPERNICUS/S1_GRD")

s1_filtered = (
    s1.filterBounds(roi)
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
      .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
      .filter(ee.Filter.eq("resolution_meters", 10))
      .select(["VV", "VH"])
)

before_image = (
    s1_filtered.filterDate(before_start, before_end)
               .median()
               .clip(roi)
)

after_image = (
    s1_filtered.filterDate(after_start, after_end)
               .median()
               .clip(roi)
)

print("Sentinel-1 images loaded ✅")

Sentinel-1 images loaded ✅


## ➗ Step 6: เพิ่มแบนด์อัตราส่วน VV/VH

อัตราส่วน `VV/VH` ช่วยเพิ่มความสามารถในการแยกน้ำออกจากพื้นผิวประเภทอื่น

In [8]:
def add_ratio_band(image):
    ratio = image.select("VV").divide(image.select("VH")).rename("VV_VH_ratio")
    return image.addBands(ratio)

before_image = add_ratio_band(before_image)
after_image = add_ratio_band(after_image)

sar_vis = {"min": [-25, -25, 0], "max": [0, 0, 2]}

Map.addLayer(before_image, sar_vis, "Before Flood (VV, VH, Ratio)", False)
Map.addLayer(after_image, sar_vis, "After Flood (VV, VH, Ratio)", False)
Map

Map(bottom=63203.0, center=[7.212625171059907, 100.49468994140626], controls=(WidgetControl(options=['position…

## 🧪 Step 7: สร้างภาพเปรียบเทียบการเปลี่ยนแปลง

เซลล์นี้ช่วยดูความต่างของค่า backscatter ก่อนและหลังน้ำท่วมแบบคร่าว ๆ

In [9]:
change_composite = ee.Image.cat([
    before_image.select("VH").rename("VH_Before"),
    after_image.select("VH").rename("VH_After"),
    before_image.select("VH").rename("VH_Before_2")
])

Map.addLayer(change_composite, {"min": -25, "max": -8}, "Change Composite", False)
Map

Map(bottom=63203.0, center=[7.212625171059907, 100.49468994140626], controls=(WidgetControl(options=['position…

## 🧼 Step 8: ฟังก์ชันแปลงหน่วยและ Refined Lee Filter

ภาพ SAR มี speckle noise ตามธรรมชาติ จึงต้องลดสัญญาณรบกวนก่อนวิเคราะห์  
ในที่นี้ใช้ **Refined Lee filter** กับแบนด์ `VH`

In [10]:
def to_natural(image):
    return ee.Image(10.0).pow(image.divide(10.0))

def to_db(image):
    return ee.Image(image).log10().multiply(10.0)

def refined_lee(img):
    # img ต้องอยู่ในหน่วย natural
    weights3 = ee.List.repeat(ee.List.repeat(1, 3), 3)
    kernel3 = ee.Kernel.fixed(3, 3, weights3, 1, 1, False)

    mean3 = img.reduceNeighborhood(ee.Reducer.mean(), kernel3)
    variance3 = img.reduceNeighborhood(ee.Reducer.variance(), kernel3)

    sample_weights = ee.List([
        [0,0,0,0,0,0,0],
        [0,1,0,1,0,1,0],
        [0,0,0,0,0,0,0],
        [0,1,0,1,0,1,0],
        [0,0,0,0,0,0,0],
        [0,1,0,1,0,1,0],
        [0,0,0,0,0,0,0]
    ])
    sample_kernel = ee.Kernel.fixed(7, 7, sample_weights, 3, 3, False)

    sample_mean = mean3.neighborhoodToBands(sample_kernel)
    sample_var = variance3.neighborhoodToBands(sample_kernel)

    gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs()
    gradients = gradients.addBands(sample_mean.select(6).subtract(sample_mean.select(2)).abs())
    gradients = gradients.addBands(sample_mean.select(3).subtract(sample_mean.select(5)).abs())
    gradients = gradients.addBands(sample_mean.select(0).subtract(sample_mean.select(8)).abs())

    max_gradient = gradients.reduce(ee.Reducer.max())
    gradmask = gradients.eq(max_gradient)
    gradmask = gradmask.addBands(gradmask)

    directions = sample_mean.select(1).subtract(sample_mean.select(4))         .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1)
    directions = directions.addBands(
        sample_mean.select(6).subtract(sample_mean.select(4))
        .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
    )
    directions = directions.addBands(
        sample_mean.select(3).subtract(sample_mean.select(4))
        .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
    )
    directions = directions.addBands(
        sample_mean.select(0).subtract(sample_mean.select(4))
        .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
    )

    directions = directions.addBands(directions.select(0).Not().multiply(5))
    directions = directions.addBands(directions.select(1).Not().multiply(6))
    directions = directions.addBands(directions.select(2).Not().multiply(7))
    directions = directions.addBands(directions.select(3).Not().multiply(8))
    directions = directions.updateMask(gradmask)
    directions = directions.reduce(ee.Reducer.sum())

    sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
    sigma_v = (
        sample_stats.toArray()
        .arraySort()
        .arraySlice(0, 0, 5)
        .arrayReduce(ee.Reducer.mean(), [0])
        .arrayGet([0])
    )

    rect_weights = ee.List.repeat(ee.List.repeat(0, 7), 3).cat(ee.List.repeat(ee.List.repeat(1, 7), 4))
    diag_weights = ee.List([
        [1,0,0,0,0,0,0],
        [1,1,0,0,0,0,0],
        [1,1,1,0,0,0,0],
        [1,1,1,1,0,0,0],
        [1,1,1,1,1,0,0],
        [1,1,1,1,1,1,0],
        [1,1,1,1,1,1,1]
    ])

    rect_kernel = ee.Kernel.fixed(7, 7, rect_weights, 3, 3, False)
    diag_kernel = ee.Kernel.fixed(7, 7, diag_weights, 3, 3, False)

    dir_mean = img.reduceNeighborhood(ee.Reducer.mean(), rect_kernel).updateMask(directions.eq(1))
    dir_var = img.reduceNeighborhood(ee.Reducer.variance(), rect_kernel).updateMask(directions.eq(1))

    dir_mean = dir_mean.addBands(img.reduceNeighborhood(ee.Reducer.mean(), diag_kernel).updateMask(directions.eq(2)))
    dir_var = dir_var.addBands(img.reduceNeighborhood(ee.Reducer.variance(), diag_kernel).updateMask(directions.eq(2)))

    for i in range(1, 4):
        dir_mean = dir_mean.addBands(
            img.reduceNeighborhood(ee.Reducer.mean(), rect_kernel.rotate(i)).updateMask(directions.eq(2*i+1))
        )
        dir_var = dir_var.addBands(
            img.reduceNeighborhood(ee.Reducer.variance(), rect_kernel.rotate(i)).updateMask(directions.eq(2*i+1))
        )
        dir_mean = dir_mean.addBands(
            img.reduceNeighborhood(ee.Reducer.mean(), diag_kernel.rotate(i)).updateMask(directions.eq(2*i+2))
        )
        dir_var = dir_var.addBands(
            img.reduceNeighborhood(ee.Reducer.variance(), diag_kernel.rotate(i)).updateMask(directions.eq(2*i+2))
        )

    dir_mean = dir_mean.reduce(ee.Reducer.sum())
    dir_var = dir_var.reduce(ee.Reducer.sum())

    var_x = dir_var.subtract(dir_mean.multiply(dir_mean).multiply(sigma_v)).divide(sigma_v.add(1.0))
    b = var_x.divide(dir_var)
    result = dir_mean.add(b.multiply(img.subtract(dir_mean)))

    return result

## 🌧️ Step 9: ใช้ Refined Lee กับแบนด์ VH และคำนวณ Change Ratio

วิธีที่ใช้คือ
\[
Change\ Ratio = \frac{After\ Flood}{Before\ Flood}
\]

ถ้าค่า ratio สูงกว่าค่าที่กำหนด จะถือว่าเป็นพื้นที่น้ำท่วมเบื้องต้น  
ในงานนี้ใช้ **threshold = 1.2**

In [11]:
before_vh_natural = to_natural(before_image.select("VH"))
after_vh_natural = to_natural(after_image.select("VH"))

before_vh_filtered = refined_lee(before_vh_natural)
after_vh_filtered = refined_lee(after_vh_natural)

change_ratio = after_vh_filtered.divide(before_vh_filtered).rename("change_ratio")

threshold = 1.2
flood_initial = change_ratio.gt(threshold).selfMask()

Map.addLayer(change_ratio, {"min": 0.8, "max": 2.0}, "Change Ratio", False)
Map.addLayer(flood_initial, {"palette": ["0000FF"]}, "Initial Flood Map", False)
Map

Map(bottom=63203.0, center=[7.212625171059907, 100.49468994140626], controls=(WidgetControl(options=['position…

## 🚫 Step 10: ตัดพื้นที่น้ำถาวรออกจากผลลัพธ์

ใช้ชุดข้อมูล **JRC Global Surface Water**  
ในที่นี้ใช้แบนด์ `seasonality` เพื่อแทนจำนวนเดือนที่พบว่ามีน้ำในแต่ละปี  
พื้นที่ที่มีน้ำอย่างน้อยประมาณ **5 เดือนขึ้นไป** จะถือเป็นน้ำถาวร/กึ่งถาวร และถูกตัดออกจากผลลัพธ์น้ำท่วม

In [12]:
seasonality = gsw.select("seasonality")
permanent_water = seasonality.gte(5)

flood_no_permanent_water = flood_initial.updateMask(permanent_water.Not())

Map.addLayer(permanent_water.selfMask(), {"palette": ["cyan"]}, "Permanent Water Mask", False)
Map.addLayer(flood_no_permanent_water, {"palette": ["blue"]}, "Flood without Permanent Water", False)
Map

Map(bottom=63203.0, center=[7.212625171059907, 100.49468994140626], controls=(WidgetControl(options=['position…

## ⛰️ Step 11: ตัดพื้นที่ลาดชันสูงกว่า 5 องศา

น้ำท่วมมักสะสมในพื้นที่ค่อนข้างราบ จึงตัดพื้นที่ที่มีความลาดชันสูงออก

In [13]:
slope = ee.Terrain.slope(srtm)
low_slope_mask = slope.lt(5)

flood_low_slope = flood_no_permanent_water.updateMask(low_slope_mask)

Map.addLayer(slope, {"min": 0, "max": 15}, "Slope", False)
Map.addLayer(flood_low_slope, {"palette": ["navy"]}, "Flood after Slope Filtering", False)
Map

Map(bottom=63203.0, center=[7.212625171059907, 100.49468994140626], controls=(WidgetControl(options=['position…

## 🔍 Step 12: ลบพิกเซลรบกวนขนาดเล็ก

ใช้ `connectedPixelCount()` เพื่อตัดกลุ่มพิกเซลที่เล็กเกินไป  
กลุ่มที่มีพิกเซลเชื่อมต่อน้อยกว่า **15 พิกเซล** จะถูกลบออก

In [14]:
connected_pixels = flood_low_slope.connectedPixelCount(maxSize=100, eightConnected=True)
flood_final = flood_low_slope.updateMask(connected_pixels.gte(15)).rename("flooded")

Map.addLayer(flood_final, {"palette": ["red"]}, "Final Flood Map")
Map

Map(bottom=63203.0, center=[7.212625171059907, 100.49468994140626], controls=(WidgetControl(options=['position…

## 📏 Step 13: คำนวณพื้นที่น้ำท่วม

คำนวณจากขนาดพิกเซลของ Sentinel-1 ที่ **10 × 10 เมตร = 100 ตารางเมตรต่อพิกเซล**

In [15]:
flood_area_image = ee.Image.pixelArea().updateMask(flood_final)

flood_area_stats = flood_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi.geometry(),
    scale=10,
    maxPixels=1e13
)

flood_area_m2 = ee.Number(flood_area_stats.get("area"))
flood_area_km2 = flood_area_m2.divide(1e6)
flood_area_rai = flood_area_m2.divide(1600)

print("Flooded area :", flood_area_m2.getInfo(), "(m²)")
print("Flooded area :", flood_area_km2.getInfo(), "(km²)")
print("Flooded area :", flood_area_rai.getInfo(), "(rai)")

Flooded area : 4080625.9823150635 (m²)
Flooded area : 4.0806259823150635 (km²)
Flooded area : 2550.3912389469147 (rai)


## 🎨 Step 14: สรุปผลบนแผนที่

ชั้นข้อมูลสำคัญที่ควรเปิดดู:
- `Songkhla Boundary`
- `Before Flood (VV, VH, Ratio)`
- `After Flood (VV, VH, Ratio)`
- `Change Ratio`
- `Final Flood Map`

In [16]:
Map

Map(bottom=63203.0, center=[7.212625171059907, 100.49468994140626], controls=(WidgetControl(options=['position…

## 💾 Step 15: ส่งออกผลลัพธ์ (Optional)
### Optional: Export ข้อมูลเชิงพื้นที่แบบ GeoTIFF ไปที่ Google Drive

ใช้เซลล์นี้เมื่ออยากได้ไฟล์ `.tif` สำหรับนำไปเปิดต่อใน QGIS, ArcGIS Pro หรือใช้วิเคราะห์เชิงพื้นที่

> GeoTIFF ตัวนี้เป็นไฟล์ข้อมูลจริงของ flood mask  

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import time
import ee

export_task = ee.batch.Export.image.toDrive(
    image=flood_final.toByte(),
    description="Songkhla_Flood_Map_2025",
    folder="Songkhla_Flooded",
    fileNamePrefix="songkhla_flood_map_2025",
    region=roi.geometry(),
    scale=10,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

export_task.start()

print("Export started ✅")
print("Output folder in Google Drive: Songkhla_Flooded")
print("Filename prefix: songkhla_flood_map_2025")

# เช็กสถานะงาน
while export_task.active():
    print("Exporting...")
    time.sleep(10)

print("Final status:", export_task.status())

Export started ✅
Output folder in Google Drive: Songkhla_Flooded
Filename prefix: songkhla_flood_map_2025
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting.

Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...
Exporting...

## 🧾 Step 16: อธิบายผลลัพธ์และข้อจำกัด

### การตีความผลลัพธ์
ผลลัพธ์สุดท้ายคือชั้นข้อมูลพื้นที่น้ำท่วมที่ได้จากการเปรียบเทียบค่า backscatter ก่อนและหลังน้ำท่วม แล้วผ่านกระบวนการกรองหลายขั้นตอน ได้แก่
1. ลด speckle noise  
2. ใช้ change ratio threshold  
3. ตัดน้ำถาวร  
4. ตัดพื้นที่ลาดชันสูง  
5. ลบกลุ่มพิกเซลรบกวนขนาดเล็ก  

### ข้อดีของวิธีนี้
- ใช้ได้ดีในช่วงที่มีเมฆมาก เพราะ SAR ไม่พึ่งพาแสง
- เหมาะกับการประเมินสถานการณ์น้ำท่วมอย่างรวดเร็ว
- ใช้งานผ่าน Google Colab และ Earth Engine ได้สะดวก

### ข้อจำกัด
- ค่า threshold 1.2 เป็นค่าที่ใช้ตามเอกสารต้นแบบ จึงอาจต้องปรับตามพื้นที่จริง
- การใช้ขอบเขตระดับจังหวัดอาจครอบคลุมพื้นที่ที่ไม่ได้รับผลกระทบโดยตรง
- พื้นที่ชุ่มน้ำหรือพื้นที่ริมน้ำบางแห่งอาจยังถูกตีความคลาดเคลื่อนได้
- ถ้ามีข้อมูลภาคสนามหรือข้อมูล flood map จากหน่วยงานภายนอก จะช่วยตรวจสอบความแม่นยำได้ดียิ่งขึ้น